In [1]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import load_model
import pickle

In [3]:
# Step 1: Load dataset
base_path = r'C:\Users\Affan computer\Face Mask Detection\Train'
categories = os.listdir(base_path)

x = []
y = []

for category in categories:
    category_path = os.path.join(base_path, category)
    if not os.path.isdir(category_path):
        continue
    for file in os.listdir(category_path):
        file_path = os.path.join(category_path, file)
        try:
            img = cv2.imread(file_path)
            if img is not None:
                img = cv2.resize(img, (224, 224))
                x.append(img)
                y.append(category)
        except Exception as e:
            print(f"Error loading {file_path}: {e}")

x = np.array(x, dtype='float32') / 255.0
y = np.array(y)

In [5]:
# Step 2: Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Show class distribution
print("Classes:", label_encoder.classes_)
print("Distribution:", np.unique(y_encoded, return_counts=True))

# Save label encoder
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

Classes: ['WithMask' 'WithoutMask']
Distribution: (array([0, 1], dtype=int64), array([5000, 5000], dtype=int64))


In [7]:
# Step 3: Train-test split
x_train, x_test, y_train, y_test = train_test_split(x, y_encoded, test_size=0.2, random_state=42)

In [9]:
# Step 4: Load VGG16 model
vgg = VGG16(include_top=False, input_shape=(224, 224, 3), pooling='avg')

In [11]:
# Step 5: Build model
model = Sequential()
for layer in vgg.layers:
    model.add(layer)
    layer.trainable = False  # Freeze pre-trained layers

model.add(Dense(1, activation='sigmoid'))

In [13]:
# Step 6: Compile and train
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=10, batch_size=32, validation_data=(x_test, y_test))

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1228s 5s/step - accuracy: 0.8178 - loss: 0.5551 - val_accuracy: 0.9335 - val_loss: 0.3138
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1211s 5s/step - accuracy: 0.9421 - loss: 0.2829 - val_accuracy: 0.9515 - val_loss: 0.2161
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1258s 5s/step - accuracy: 0.9524 - loss: 0.1994 - val_accuracy: 0.9560 - val_loss: 0.1722
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1208s 5s/step - accuracy: 0.9585 - loss: 0.1620 - val_accuracy: 0.9600 - val_loss: 0.1477
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1327s 5s/step - accuracy: 0.9616 - loss: 0.1406 - val_accuracy: 0.9630 - val_loss: 0.1301
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1328s 5s/step - accuracy: 0.9658 - loss: 0.1252 - val_accuracy: 0.9675 - val_loss: 0.1182
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1357s 5s/step - accuracy: 0.9699 - loss: 0.1145 - val_accuracy: 0.9670 - val_loss: 0.1094
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1366s 5s/step - accuracy: 0.9707 - loss: 0.1060 - 

In [15]:
# Step 7: Save model
model.save("face_mask_model.h5")
print(" Model and encoder saved successfully.")

 Model and encoder saved successfully.


In [17]:
# Load trained model and label encoder
model = load_model("face_mask_model.h5")
with open("label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

In [19]:
# Load OpenCV face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

# Function to detect mask on detected face
def detect_face_mask(face_img):
    img = cv2.resize(face_img, (224, 224))
    img = img.astype('float32') / 255.0
    img = np.expand_dims(img, axis=0)

    prediction = model.predict(img)[0][0]
    if prediction >= 0.5:
        label = "No Mask"
        confidence = prediction
    else:
        label = "Mask"
        confidence = 1 - prediction

    return f"{label} ({confidence*100:.2f}%)", label

In [22]:
# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4)

    for (x, y, w, h) in faces:
        face_img = frame[y:y+h, x:x+w]
        result_text, label = detect_face_mask(face_img)

        color = (0, 255, 0) if "Mask" in result_text else (0, 0, 255)
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, result_text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    cv2.imshow("Face Mask Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('x'):
        break

cap.release()
cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 282ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 458ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 425ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 356ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 254ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 268ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 286ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 303ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 284ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 268ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 285ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 285ms/step
